# B03 — Performance Tuning and Optimization

**Time: about 60 minutes.**
Covers: Performance Tuning Optimization

### What you will be able to do afterwards

- Measure a query's cost before changing anything.
- Count the files a query actually touches, and reduce that number.
- Read an execution plan well enough to spot a broadcast and a shuffle.
- Explain when clustering helps and when it does nothing.

### The rule for this notebook

**No change without a measurement on both sides of it.** Every step here records a number
before and after. "It felt faster" is not an outcome.

In [ ]:
from pyspark.sql import DataFrame, functions as F
from helpers import utils, event_stream

cfg = utils.get_configs("events_wide")
wide_table = cfg["table_silver"]
tuning_log = utils.get_configs("tuning_log")["table_gold"]
replay_dir = event_stream.replay_path()

spark = utils.spark
print(wide_table)
print(tuning_log)
print(replay_dir)

## Step 1 — Build something big enough to measure

The replay JSON files from setup make a convenient source. Read them as a **batch** — no
streaming in this notebook — and amplify them until the table is large enough that layout
matters.

**TO DO**

1. Read every file in `replay_dir` with `spark.read.json`.
2. Amplify to at least 500,000 rows. A cross join against a small range works:
   `df.crossJoin(spark.range(100))`, then make `event_id` unique again.
3. Write the result as `events_wide` in your silver schema. Write it **badly on purpose**:
   many small files. `df.repartition(200).write...` will do it.

> **Question:** you just deliberately created 200 files for a table this size. Before you
> measure anything — predict what that will cost, and which of the numbers in Step 2 will
> show it.

In [ ]:
# TO DO: read the replay files as batch


# TO DO: amplify to 500k+ rows with unique event_ids


# TO DO: write as events_wide with many small files

## Step 2 — Measure the baseline

**TO DO**

Create `tuning_log` in gold with columns `stage`, `metric_time`, `num_files`,
`size_bytes`, `files_scanned`, `duration_ms`, `note`.

Then write `measure(stage, note)` that records:

- `num_files` and `size_bytes` from `DESCRIBE DETAIL`.
- `files_scanned` — the number of files a selective query actually reads:

  ```sql
  SELECT COUNT(DISTINCT _metadata.file_path)
  FROM events_wide
  WHERE product_id = 7
  ```

  `_metadata.file_path` is the file each row came from, so counting the distinct values
  under a filter tells you exactly how many files the scan had to open. This is the number
  that data skipping changes.

- `duration_ms` — wall-clock time for that same query, via `time.perf_counter()`.

Call `measure("before", ...)`.

> **Question:** run the timing twice. The second run is faster. Why, and what does that
> mean for how you should benchmark?

In [ ]:
import time

PROBE_SQL = f"SELECT COUNT(*) FROM {wide_table} WHERE product_id = 7"
FILES_SQL = f"SELECT COUNT(DISTINCT _metadata.file_path) AS files FROM {wide_table} WHERE product_id = 7"


def measure(stage: str, note: str = "") -> dict:
    """
    Record num_files, size_bytes, files_scanned and duration_ms for the current table state.

    Args:
        stage: 'before' or 'after'.
        note: free text, e.g. which change you just made.
    Returns:
        The measurement, also appended to tuning_log.
    """
    # TO DO
    pass


# TO DO: create tuning_log, then measure("before")

## Step 3 — Change the layout, then measure again

**TO DO**

1. Look at the probe query. Which column does it filter on? That is your clustering key.
2. `ALTER TABLE ... CLUSTER BY (...)`.
3. `OPTIMIZE`. Enabling clustering does not move existing data; `OPTIMIZE` is what applies it.
4. `measure("after", ...)`.
5. Print the before and after rows side by side and compute the ratio for both
   `files_scanned` and `duration_ms`.

> **Questions:**
> - Which improved more, files scanned or wall-clock time? Why might they not move together?
> - Now cluster on `device_type` instead and re-measure. Three distinct values across
>   500,000 rows. Predict the result first, then check.

In [ ]:
# TO DO: cluster, optimize, measure("after")


# TO DO: compare before and after

## Step 4 — Read a plan

**TO DO**

1. Join `events_wide` to the small products table from your silver schema.
2. Run `.explain("formatted")` and find in the output:
   - `BroadcastHashJoin` or `SortMergeJoin` — which did Spark choose?
   - `PhotonScan` / file scan node — how many files does it report?
   - Any `AQEShuffleRead` node — adaptive query execution rewriting the plan at runtime.
3. Force the other join strategy with `F.broadcast()` or by disabling the auto-broadcast
   threshold, and compare plans.

**Tips**

- Auto-broadcast threshold: `spark.conf.get("spark.sql.autoBroadcastJoinThreshold")`.
- Setting it to `-1` disables broadcasting entirely. Set it back afterwards.

> **Questions:**
> - Broadcasting sends the small side to every executor. What makes that a good trade, and
>   at what size does it stop being one?
> - AQE can switch a sort-merge join to a broadcast join *after* the query starts. What
>   does it know at that point that the optimizer didn't?

In [ ]:
# TO DO: join to products, explain the plan


# TO DO: force the other strategy and compare

## Step 5 — What not to do

**TO DO**

Argue, in a markdown cell, against each of these. One paragraph each. They are all things
somebody will suggest in a review.

1. "Partition `events_wide` by `event_id` — it's unique, so pruning will be perfect."
2. "Run `OPTIMIZE` after every write."
3. "Set `spark.sql.shuffle.partitions` to 2000 because the table is big."
4. "Cache the table — it's queried a lot."

In [ ]:
# TO DO: write your answers in a markdown cell, or as comments here

## Checks

In [ ]:
from helpers import test_runner

test_runner.run("B03-performance-tuning")

## Recap

- `COUNT(DISTINCT _metadata.file_path)` under a filter is the cheapest honest measure of
  data skipping you have.
- Clustering only helps for columns your queries actually filter on, and only after
  `OPTIMIZE` has rewritten the data.
- Wall-clock time is noisy — caching, warm pools, cluster contention. File counts are not.
- Reading a plan is a skill you build by looking at plans you already understand the
  answer to.